# 6.21 Gradient checking

Gradient checking compares backprop to finite differences so silent derivative bugs become measurable.

Backprop is only useful when its derivatives match the scalar loss being optimized; centered finite differences make that agreement measurable. Save a copy to Drive to edit.

In [ ]:

import math
import time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

np.random.seed(42)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def pad_to_64(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 64:
        return X.copy()
    if X.shape[1] > 64:
        return X[:, :64].copy()
    out = np.zeros((X.shape[0], 64), dtype=float)
    out[:, :X.shape[1]] = X
    return out


def split_scale(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te




def softmax_logits(X, W):
    logits = X @ W
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def train_small_mlp_loss(X, y, random_state=0, max_iter=90):
    x_tr, x_te, y_tr, y_te = split_scale(X, y)
    clf = MLPClassifier(hidden_layer_sizes=(16,), activation="relu", solver="adam", alpha=0.001, max_iter=max_iter, random_state=random_state)
    clf.fit(x_tr, y_tr)
    proba = clf.predict_proba(x_te)
    loss = log_loss(y_te, proba, labels=clf.classes_)
    acc = accuracy_score(y_te, clf.predict(x_te))
    return float(loss), float(acc), clf


def train_small_mlp_acc(X, y, random_state=0, max_iter=90):
    loss, acc, clf = train_small_mlp_loss(X, y, random_state=random_state, max_iter=max_iter)
    return acc, loss, clf


def preview_ladder(rungs):
    for rung, (name, X, y) in enumerate(rungs, 1):
        classes = np.unique(y)
        print(f"D{rung}: {name:36s} X={X.shape} classes={len(classes)} sample_y={classes[:5].tolist()}")


def plot_metric_table(rows, metric_name):
    print(f"{'rung':<4} {'dataset':<36} {metric_name:>10}")
    for row in rows:
        print(f"{row['rung']:<4} {row['name']:<36} {row['metric']:10.4f}")


def plot_summary(rows, metric_name, ylabel):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    names = [row["rung"] for row in rows]
    values = [row["metric"] for row in rows]
    axes[0].bar(names, values, color="steelblue")
    axes[0].set_title("Metric by ladder rung")
    axes[0].set_ylabel(ylabel)
    axes[1].plot(names, values, marker="o")
    axes[1].set_title("D1→D5 trend")
    axes[1].set_ylabel(ylabel)
    fig.tight_layout()
    plt.show()


def show_artifacts(rungs, title):
    fig, axes = plt.subplots(1, len(rungs), figsize=(13, 2.6))
    for ax, (name, X, y) in zip(axes, rungs):
        if X.shape[1] == 64:
            ax.imshow(X[0].reshape(8, 8), cmap="gray")
            ax.set_title(name.split()[0])
        else:
            ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="tab10")
            ax.set_title(name.split()[0])
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


## The concept, built once

For one parameter, centered checking uses $\frac{L(\theta_i+h)-L(\theta_i-h)}{2h}$. The lesson scalar pass uses $1.2\cdot1.5 + (-0.4)\cdot(-0.5)+0.6=2.6$ and the update $2.0-0.06\cdot1.2=1.928$.

In [ ]:

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def logistic_loss(theta, x, y):
    z = float(np.dot(theta[:2], x) + theta[2])
    p = sigmoid(z)
    eps = 1e-12
    return float(-(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps)))


def analytic_grad(theta, x, y):
    z = float(np.dot(theta[:2], x) + theta[2])
    p = sigmoid(z)
    return np.r_[((p - y) * x), p - y]


def numeric_grad(theta, x, y, h=1e-5):
    grad = np.zeros_like(theta)
    for i in range(len(theta)):
        step = np.zeros_like(theta)
        step[i] = h
        plus = logistic_loss(theta + step, x, y)
        minus = logistic_loss(theta - step, x, y)
        grad[i] = (plus - minus) / (2 * h)
    return grad


def gradient_check(theta, x, y):
    numeric = numeric_grad(theta, x, y)
    analytic = analytic_grad(theta, x, y)
    rel = np.linalg.norm(numeric - analytic) / (np.linalg.norm(numeric) + np.linalg.norm(analytic))
    return numeric, analytic, float(rel)

x = np.array([1.5, -0.5])
theta = np.array([1.2, -0.4, 0.6])
y = 1.0
score = float(np.dot(theta[:2], x) + theta[2])
updated = 2.0 - 0.06 * 1.2
numeric, analytic, rel = gradient_check(theta, x, y)
print("score", score)
print("updated parameter", updated)
print("numeric", numeric)
print("analytic", analytic)
print("relative error", rel)
assert np.isclose(score, 2.6)
assert np.isclose(updated, 1.928)
assert rel < 1e-10


The reusable checker flattens all parameters, calls the loss twice per coordinate, and compares against the analytic gradient with relative error.

In [ ]:
print('D1 centered finite difference matches analytic backprop:', rel < 1e-10)

## The dataset ladder

The same code is run from a four-point XOR problem through real 8×8 digit images and a noisy shifted D5 variant.

In [ ]:
rungs = clf_digits_ladder()
preview_ladder(rungs)
show_artifacts(rungs, 'Ladder preview')

## Run the SAME method across D1–D5

We gradient-check a real logistic-loss coordinate on each rung, then train a tiny MLP and report held-out loss.

In [ ]:

rows = []
for idx, (name, X, y) in enumerate(rungs, 1):
    X64 = pad_to_64(X)
    x_tr, x_te, y_tr, y_te = split_scale(X64, y)
    binary_y = (y_tr == y_tr[0]).astype(float)
    theta0 = np.linspace(-0.03, 0.03, x_tr.shape[1] + 1)
    sample_x = x_tr[0]
    num, ana, rung_rel = gradient_check(theta0[:3], sample_x[:2], binary_y[0])
    loss, acc, clf = train_small_mlp_loss(X64, y, random_state=idx)
    rows.append({"rung": f"D{idx}", "name": name, "metric": loss, "rel": rung_rel})
plot_metric_table(rows, "loss")
for row in rows:
    print(row["rung"], "gradient relative error", f"{row['rel']:.2e}")


## Results visualization

In [ ]:
show_artifacts(rungs, 'Gradient-checking artifacts')
plot_summary(rows, 'loss', 'held-out loss')

## Pitfall on D5: a silent derivative bug

A shape/sign bug can still produce numbers. The fix is to print relative error before trusting D5 training.

In [ ]:

name, X, y = rungs[-1]
X64 = pad_to_64(X)
x_tr, x_te, y_tr, y_te = split_scale(X64, y)
binary_y = (y_tr == 0).astype(float)
theta = np.array([0.1, -0.2, 0.05])
x = x_tr[0, :2]
correct_numeric, correct_analytic, good_rel = gradient_check(theta, x, binary_y[0])
bad_analytic = correct_analytic.copy()
bad_analytic[0] = -bad_analytic[0]
bad_rel = np.linalg.norm(correct_numeric - bad_analytic) / (np.linalg.norm(correct_numeric) + np.linalg.norm(bad_analytic))
print("buggy relative error", bad_rel)
print("fixed relative error", good_rel)
assert bad_rel > 0.05
assert good_rel < 1e-10


## Evaluate it + Practice

- Metric: held-out loss; compare it with a majority-class or plain logistic baseline.
- Sanity check: D1 should be hand-checkable before trusting D5.
- Ablation: turn off the key idea and the reported metric should get worse or the pitfall should reappear.
- Failure signals: unstable losses, collapsed routing, inflated gradient error, or a D5 result that ignores label noise.

Practice 1: change one hyperparameter and rerun the D1 assert before touching D5.

Practice 2: add a baseline row to the ladder table and explain the largest gap.

Practice 3: make the D5 pitfall more severe, then show the same fix still helps.